# 01 — Teacher Training (M1)
**Knowledge Distillation + Quantization for Compact Models**

Fine-tune `bert-base-uncased` on GLUE SST-2 to produce **M1 — the FP32 teacher model**.

| Step | Status |
|------|--------|
| Dataset loading | ✅ Implemented (this notebook, Section A) |
| Teacher training | 🔜 Phase 3 |

---
### Prerequisites
1. Run `notebooks/00_dataset_preparation.ipynb` on Colab first.
2. Download `data/processed/` and place it in your local project.
3. This notebook can run either on Colab (for training) or locally (for data loading verification).


## Section A — Environment Setup

In [ ]:
# ── Runtime detection ────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')
print(f'Python: {sys.version}')

if IN_COLAB:
    !pip install -q transformers>=4.38.0 datasets>=2.18.0 accelerate>=0.27.0
    # Set PROJECT_ROOT after cloning or mounting Drive
    # !git clone https://github.com/YOUR_USERNAME/knowledge-distillation-quantization.git /content/kdq
    PROJECT_ROOT = '/content/kdq'  # adjust if needed
else:
    PROJECT_ROOT = str(Path('..').resolve())

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

In [ ]:
# ── Imports and configuration ────────────────────────────────────────────────
import torch
from src import config

config.print_config()
print(f'\nDevice : {config.DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## Section B — Dataset Loading

Two loading paths are supported:

| Environment | Method | Requires |
|-------------|--------|----------|
| Google Colab | `ds.load_sst2()` | `datasets` + `pyarrow` |
| Local (Windows WDAC) | `ds.load_processed_local()` | Only `pandas` + `transformers` |

The cell below auto-selects the correct path.

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
import src.dataset as ds

tokenizer = ds.get_tokenizer(config.TEACHER_MODEL_NAME)
print(f'Tokenizer loaded: {tokenizer.__class__.__name__}  vocab_size={tokenizer.vocab_size:,}')

# Try HuggingFace datasets first (Colab), fall back to local Parquet (local)
try:
    dataset   = ds.load_sst2(subset_size=config.SUBSET_SIZE, seed=config.RANDOM_SEED)
    tokenized = ds.tokenize_dataset(dataset, tokenizer, config.MAX_SEQ_LENGTH)
    train_loader, val_loader = ds.get_dataloaders(
        tokenized, tokenizer, batch_size=config.BATCH_SIZE
    )
    print('Data loaded via HuggingFace datasets.')
except Exception as hf_err:
    print(f'HF datasets unavailable ({type(hf_err).__name__}). Using local processed files.')
    PROCESSED_DIR = Path(PROJECT_ROOT) / 'data' / 'processed'
    train_df, val_df, metadata = ds.load_processed_local(PROCESSED_DIR)
    train_loader, val_loader = ds.get_dataloaders(
        (train_df, val_df), tokenizer, batch_size=config.BATCH_SIZE
    )
    print(f'Data loaded from {PROCESSED_DIR}')

print(f'\nTrain batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')

In [ ]:
# ── Inspect one batch ────────────────────────────────────────────────────────
batch = next(iter(train_loader))
print('Batch keys        :', list(batch.keys()))
print('input_ids shape   :', batch['input_ids'].shape)
print('attention_mask    :', batch['attention_mask'].shape)
print('labels            :', batch['labels'][:8])

## Section C — Teacher Model Training
**Phase 3 placeholder** — training cells implemented in Phase 3.

In [ ]:
# TODO (Phase 3): instantiate BertForSequenceClassification, run training loop,
#                 evaluate on val_loader, save checkpoint to models/teacher/
print('Teacher training — to be implemented in Phase 3.')